# Passo Firme — melhorias sem novos dados

O modelo final (`modelos/best.pt`, YOLO26n, 3 classes, 640 px) tem mAP50 de 0,16 no teste com 31 fotos de São Paulo. Este notebook testa, **no mesmo conjunto de teste**, quatro jeitos baratos de melhorar sem anotar fotos novas:

| | Experimento | O que muda | Custo |
| --- | --- | --- | --- |
| E0 | Base | modelo final, imagem de 640 px | — |
| E1 | Resolução maior na inferência | o mesmo modelo lê a foto em 960 px | sem treino |
| E2 | Resolução maior na inferência | 1280 px | sem treino |
| E3 | Recortes (*tiles*) | a foto é dividida em 4 pedaços com sobreposição, mais a foto inteira, e as caixas são juntadas | sem treino |
| E4 | Treino + validação | treina de novo com as 254 imagens de treino e validação juntas (38 a mais), pelo mesmo número de épocas | ~6 min de GPU |
| E5 | E4 + recortes | o modelo do E4 com os recortes do E3 | — |

Todos são medidos com o mesmo código: mAP50 por classe (a mesma conta do Ultralytics), precisão e recall com confiança mínima de 0,25 (o limiar do serviço) e, por foto, quantas fotos com barreira recebem ao menos uma detecção. O teste não entra em nenhum treino.

**Antes de rodar:** *Ambiente de execução → Alterar o tipo* → **GPU T4**; segredo `ROBOFLOW_API_KEY` ligado. Depois, *Executar tudo* (uns 15 minutos).

## 1. Instalação, dataset e modelo final

In [1]:
%pip install -q "ultralytics>=8.4.0" roboflow

In [2]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace("pedros-workspace-md4aw").project("passo-firme-calcadas").version(1).download("yolov8", location="/content/dataset")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/dataset in yolov8:: 100%|██████████| 575/575 [00:00<00:00, 946.98it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
!wget -q -O /content/best.pt https://github.com/predowss/passo-firme-visao/raw/main/modelos/best.pt
!nvidia-smi --query-gpu=name --format=csv,noheader

Tesla T4


## 2. Funções de avaliação

mAP50 com interpolação de 101 pontos, como no Ultralytics; contagem de acertos num limiar fixo; contagem por foto; junção das caixas dos recortes (NMS por classe).

In [4]:
import numpy as np

_trapezio = getattr(np, "trapezoid", None) or np.trapz


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / uniao if uniao > 0 else 0.0


def ap50(casos, nomes, iou_min=0.5):
    """AP por classe com IoU >= iou_min, interpolação de 101 pontos (a mesma conta do Ultralytics).
    casos: lista de (reais, previstas); reais = [(classe, x1, y1, x2, y2)];
    previstas = [(classe, confianca, x1, y1, x2, y2)]."""
    resultado = {}
    for c, nome in enumerate(nomes):
        previsoes, total_reais = [], 0
        for i, (reais, prev) in enumerate(casos):
            total_reais += sum(1 for r in reais if r[0] == c)
            previsoes += [(p[1], i, p[2:]) for p in prev if p[0] == c]
        if total_reais == 0:
            resultado[nome] = None
            continue
        previsoes.sort(key=lambda p: -p[0])
        usados, acertos = {}, []
        for _, i, caixa in previsoes:
            reais = [r for r in casos[i][0] if r[0] == c]
            melhor, melhor_iou = None, iou_min
            for j, r in enumerate(reais):
                if j in usados.get(i, set()):
                    continue
                v = iou(r[1:], caixa)
                if v >= melhor_iou:
                    melhor, melhor_iou = j, v
            if melhor is None:
                acertos.append(0)
            else:
                usados.setdefault(i, set()).add(melhor)
                acertos.append(1)
        tp = np.cumsum(acertos) if acertos else np.zeros(0)
        fp = np.cumsum([1 - a for a in acertos]) if acertos else np.zeros(0)
        recall = tp / total_reais
        precisao = tp / np.maximum(tp + fp, 1e-9)
        mrec = np.concatenate(([0.0], recall, [1.0]))
        mpre = np.concatenate(([1.0], precisao, [0.0]))
        mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
        x = np.linspace(0, 1, 101)
        resultado[nome] = float(_trapezio(np.interp(x, mrec, mpre), x))
    validos = [v for v in resultado.values() if v is not None]
    resultado["todas"] = float(np.mean(validos)) if validos else None
    return resultado


def contar(casos, nomes, limiar, iou_min=0.5):
    """Acertos, falsos alarmes e barreiras não encontradas num limiar fixo de confiança."""
    tp = fp = fn = 0
    for reais, previstas in casos:
        prev = sorted([p for p in previstas if p[1] >= limiar], key=lambda p: -p[1])
        livres = list(range(len(reais)))
        for p in prev:
            melhor, melhor_iou = None, iou_min
            for j in livres:
                if reais[j][0] == p[0]:
                    v = iou(reais[j][1:], p[2:])
                    if v >= melhor_iou:
                        melhor, melhor_iou = j, v
            if melhor is None:
                fp += 1
            else:
                tp += 1
                livres.remove(melhor)
        fn += len(livres)
    return tp, fp, fn


def por_foto(casos, limiar):
    com = [prev for reais, prev in casos if reais]
    sem = [prev for reais, prev in casos if not reais]
    achou = sum(1 for prev in com if any(p[1] >= limiar for p in prev))
    alarme = sum(1 for prev in sem if any(p[1] >= limiar for p in prev))
    return achou, len(com), alarme, len(sem)


def nms_por_classe(caixas, iou_max=0.5):
    """caixas: [(classe, conf, x1, y1, x2, y2)] -> sem duplicatas (mesma classe, IoU > iou_max)."""
    ficam = []
    for cx in sorted(caixas, key=lambda b: -b[1]):
        if all(not (f[0] == cx[0] and iou(f[2:], cx[2:]) > iou_max) for f in ficam):
            ficam.append(cx)
    return ficam


def janelas(w, h, colunas=2, linhas=2, sobra=0.2):
    """Recortes que cobrem a imagem com sobreposição: [(x1, y1, x2, y2)]."""
    tw = int(w / (colunas - (colunas - 1) * sobra))
    th = int(h / (linhas - (linhas - 1) * sobra))
    xs = [round(i * (w - tw) / max(1, colunas - 1)) for i in range(colunas)]
    ys = [round(j * (h - th) / max(1, linhas - 1)) for j in range(linhas)]
    return [(x, y, x + tw, y + th) for y in ys for x in xs]

In [5]:
import time
import cv2
import yaml
from pathlib import Path
from ultralytics import YOLO

base = Path(dataset.location)
nomes = yaml.safe_load((base / "data.yaml").read_text())["names"]
imagens = sorted((base / "test" / "images").glob("*"))

def caixas_reais(img_path, w, h):
    f = base / "test" / "labels" / (img_path.stem + ".txt")
    reais = []
    if f.exists():
        for linha in f.read_text().splitlines():
            if linha.strip():
                c, xc, yc, bw, bh = map(float, linha.split()[:5])
                reais.append((int(c), (xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h))
    return reais

def prever(modelo, img, imgsz=640, desloc=(0, 0)):
    r = modelo.predict(img, conf=0.001, imgsz=imgsz, verbose=False)[0]
    dx, dy = desloc
    return [(int(c), float(cf), x1 + dx, y1 + dy, x2 + dx, y2 + dy)
            for (x1, y1, x2, y2), c, cf in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]

def prever_recortes(modelo, img, imgsz=640):
    h, w = img.shape[:2]
    caixas = prever(modelo, img, imgsz)                      # a foto inteira pega o que é grande
    for x1, y1, x2, y2 in janelas(w, h):                     # os recortes pegam o que é pequeno
        caixas += prever(modelo, np.ascontiguousarray(img[y1:y2, x1:x2]), imgsz, (x1, y1))
    return nms_por_classe(caixas, 0.5)

def avaliar(nome, modelo, modo="normal", imgsz=640):
    casos, inicio = [], time.perf_counter()
    for p in imagens:
        img = cv2.imread(str(p))
        h, w = img.shape[:2]
        prev = prever_recortes(modelo, img, imgsz) if modo == "recortes" else prever(modelo, img, imgsz)
        casos.append((caixas_reais(p, w, h), prev))
    ms = (time.perf_counter() - inicio) * 1000 / len(imagens)
    ap = ap50(casos, nomes)
    tp, fp, fn = contar(casos, nomes, 0.25)
    achou, com, alarme, sem = por_foto(casos, 0.25)
    linha = {"experimento": nome, "mAP50": ap["todas"], **{f"mAP50 {k}": ap[k] for k in nomes},
             "precisão (0,25)": tp / max(tp + fp, 1), "recall (0,25)": tp / max(tp + fn, 1),
             "fotos com barreira detectadas (0,25)": f"{achou} de {com}", "alarme em fotos sem barreira": f"{alarme} de {sem}",
             "ms por imagem (GPU)": ms}
    print(nome, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in linha.items() if k != "experimento"})
    return linha

final = YOLO("/content/best.pt")
linhas = []

## 3. E0 a E3: o modelo final, sem treinar de novo

O E0 deve dar perto do mAP50 de 0,159 que o Ultralytics mediu: confirma que a conta está certa.

In [6]:
linhas.append(avaliar("E0 base (640)", final))
linhas.append(avaliar("E1 inferência em 960", final, imgsz=960))
linhas.append(avaliar("E2 inferência em 1280", final, imgsz=1280))
linhas.append(avaliar("E3 recortes 2x2 + foto inteira", final, modo="recortes"))

E0 base (640) {'mAP50': 0.175, 'mAP50 buraco': 0.076, 'mAP50 desnivel': 0.114, 'mAP50 obstaculo': 0.334, 'precisão (0,25)': 0.349, 'recall (0,25)': 0.211, 'fotos com barreira detectadas (0,25)': '21 de 26', 'alarme em fotos sem barreira': '2 de 5', 'ms por imagem (GPU)': 513.094}
E1 inferência em 960 {'mAP50': 0.131, 'mAP50 buraco': 0.074, 'mAP50 desnivel': 0.051, 'mAP50 obstaculo': 0.269, 'precisão (0,25)': 0.261, 'recall (0,25)': 0.169, 'fotos com barreira detectadas (0,25)': '20 de 26', 'alarme em fotos sem barreira': '3 de 5', 'ms por imagem (GPU)': 26.134}
E2 inferência em 1280 {'mAP50': 0.115, 'mAP50 buraco': 0.095, 'mAP50 desnivel': 0.092, 'mAP50 obstaculo': 0.157, 'precisão (0,25)': 0.209, 'recall (0,25)': 0.127, 'fotos com barreira detectadas (0,25)': '21 de 26', 'alarme em fotos sem barreira': '3 de 5', 'ms por imagem (GPU)': 29.719}
E3 recortes 2x2 + foto inteira {'mAP50': 0.204, 'mAP50 buraco': 0.084, 'mAP50 desnivel': 0.285, 'mAP50 obstaculo': 0.244, 'precisão (0,25)': 0.2

## 4. E4: treino de novo com treino + validação juntos

Mesmas configurações do modelo final (YOLO26n do COCO, 640 px, lote 16, semente 42) e 50 épocas fixas (o modelo final teve o melhor resultado na época 46). Como a validação entra no treino, não há escolha de época: vale o modelo da última.

In [7]:
cfg = yaml.safe_load((base / "data.yaml").read_text())
cfg["path"] = str(base)
cfg["train"] = [str(base / "train" / "images"), str(base / "valid" / "images")]
cfg["val"] = str(base / "valid" / "images")
cfg["test"] = str(base / "test" / "images")
Path("/content/data_trainval.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True))

YOLO("yolo26n.pt").train(data="/content/data_trainval.yaml", imgsz=640, epochs=50, batch=16, seed=42,
                         val=False, project="/content/runs", name="E4_trainval", exist_ok=True, plots=False)
novo = YOLO("/content/runs/E4_trainval/weights/last.pt")
linhas.append(avaliar("E4 treino + validação (640)", novo))
linhas.append(avaliar("E5 treino + validação + recortes", novo, modo="recortes"))

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_trainval.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E4_trainval, nbs=64, nms=None, opset=None, optimize=False, opt

## 5. Comparação

In [8]:
import pandas as pd
tabela = pd.DataFrame(linhas).round(3)
tabela

,experimento,mAP50,mAP50 buraco,mAP50 desnivel,mAP50 obstaculo,"precisão (0,25)","recall (0,25)","fotos com barreira detectadas (0,25)",alarme em fotos sem barreira,ms por imagem (GPU)
0,E0 base (640),0.175,0.076,0.114,0.334,0.349,0.211,21 de 26,2 de 5,513.094
1,E1 inferência em 960,0.131,0.074,0.051,0.269,0.261,0.169,20 de 26,3 de 5,26.134
2,E2 inferência em 1280,0.115,0.095,0.092,0.157,0.209,0.127,21 de 26,3 de 5,29.719
3,E3 recortes 2x2 + foto inteira,0.204,0.084,0.285,0.244,0.245,0.324,26 de 26,2 de 5,72.331
4,E4 treino + validação (640),0.157,0.063,0.158,0.248,0.312,0.141,20 de 26,1 de 5,26.235
5,E5 treino + validação + recortes,0.113,0.108,0.035,0.195,0.158,0.169,23 de 26,4 de 5,73.076


## 6. Baixar a tabela e o modelo do E4

`melhorias.csv` vai para o relatório. O `best_trainval.pt` só substitui o modelo final se o E4 for claramente melhor (e depois de um teste ao vivo).

In [9]:
import shutil
from google.colab import files

tabela.to_csv("/content/melhorias.csv", index=False)
shutil.copy("/content/runs/E4_trainval/weights/last.pt", "/content/best_trainval.pt")
files.download("/content/melhorias.csv")
files.download("/content/best_trainval.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resumo desta execução (17/11/2026, GPU T4)

- **E0 confere a conta:** precisão 0,349, recall 0,211, 21 de 26 fotos com detecção e alarme em 2 de 5, iguais ao notebook 03 (15 acertos, 28 falsos alarmes, 56 não encontradas). O mAP50 (0,175) fica acima do 0,159 do Ultralytics por detalhes de método; por isso cada experimento é comparado com o E0. O tempo do E0 (513 ms) inclui o aquecimento da GPU.
- **Resolução maior piorou:** mAP50 de 0,131 em 960 px e 0,115 em 1280 px. O modelo aprendeu os objetos no tamanho que vê a 640 px.
- **Recortes (E3) foram o único ganho:** mAP50 de 0,175 para 0,204, recall de 0,21 para 0,32 e 26 de 26 fotos com barreira com detecção. Em troca, a precisão caiu de 0,35 para 0,25, obstáculo caiu de 0,33 para 0,24 e o tempo subiu de cerca de 26 para 72 ms por foto. O ganho em desnível (0,11 para 0,29) vem de só 10 barreiras.
- **Treino + validação (E4) não ajudou:** mAP50 de 0,157, recall de 0,14. O `best_trainval.pt` não foi adotado. O E5 (E4 + recortes) ficou em 0,113.
- **Decisão:** os recortes viraram a opção `recortes=true` do `/detectar` (desligada por padrão) e ficaram fora do tempo real. Nenhuma das ideias resolveu buraco: o gargalo são os dados.
